In [1]:
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import confusion_matrix

In [2]:
rf = joblib.load("models/random_forest_unknown.pkl")
ocsvm = joblib.load("models/ocsvm_unknown.pkl")

X_unknown = np.load("processed/x_test_unknown.npy")
y_unknown = np.load("processed/y_test_unknown.npy")

In [3]:
print(type(rf))
print(type(ocsvm))

print(X_unknown.shape)

<class 'sklearn.ensemble._forest.RandomForestClassifier'>
<class 'sklearn.svm._classes.OneClassSVM'>
(25122, 78)


In [4]:
# ============================================
# Cell 3 : Extract Random Forest False Negatives
# ============================================

# RF predictions
rf_pred = rf.predict(X_unknown)

# False Negatives
fn_mask = (y_unknown == 1) & (rf_pred == 0)

X_fn = X_unknown[fn_mask]
y_fn = y_unknown[fn_mask]

print("=" * 60)
print("Random Forest False Negatives")
print("=" * 60)

print("Total False Negatives :", len(X_fn))

Random Forest False Negatives
Total False Negatives : 10715


In [5]:
# ============================================
# Cell 4 : OCSVM on RF False Negatives
# ============================================

# OCSVM prediction
ocsvm_pred = ocsvm.predict(X_fn)

# Convert prediction
# OCSVM:
#  1  -> Normal
# -1  -> Anomaly

ocsvm_binary = np.where(ocsvm_pred == -1, 1, 0)

print("=" * 60)
print("OCSVM Prediction Completed")
print("=" * 60)

print("Prediction Distribution")

unique, counts = np.unique(ocsvm_binary, return_counts=True)

for label, count in zip(unique, counts):
    print(f"Class {label}: {count}")

OCSVM Prediction Completed
Prediction Distribution
Class 0: 4477
Class 1: 6238


In [6]:
# ============================================
# Cell 5 : Recovery Analysis
# ============================================

recovered = (ocsvm_binary == 1).sum()

recovery_rate = recovered / len(y_fn) * 100

print("=" * 60)
print("OCSVM Recovery Analysis")
print("=" * 60)

print(f"RF False Negatives : {len(y_fn)}")

print(f"Recovered by OCSVM : {recovered}")

print(f"Recovery Rate : {recovery_rate:.2f}%")

OCSVM Recovery Analysis
RF False Negatives : 10715
Recovered by OCSVM : 6238
Recovery Rate : 58.22%
